# Time Series Forecasting with ProphetForecaster

This notebook demonstrates the **ProphetForecaster** on a synthetic daily time series with:

1. Linear upward trend
2. Yearly and weekly seasonality
3. 30-day ahead forecast with confidence bands
4. Trend decomposition

The forecaster uses Facebook Prophet when available and falls back to a linear trend + seasonality model otherwise.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from anomalykit import ProphetForecaster
from generate_data import generate_forecast_data

## Data Generation

365 daily data points: linear trend (0.15/day) + yearly seasonality (amplitude 20) + weekly seasonality (amplitude 5) + noise.

In [ ]:
df = generate_forecast_data(n=365, seed=42)

print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Value range: [{df['y'].min():.1f}, {df['y'].max():.1f}]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["ds"], df["y"], lw=0.8, color="steelblue")
ax.set_title("Synthetic Daily Time Series")
ax.set_ylabel("Value")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Fit and Forecast

In [ ]:
forecaster = ProphetForecaster(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95,
)

forecaster.fit(df)

horizon = 30
result = forecaster.predict(periods=horizon, freq="D")

print(f"Trend: {result.trend}")
print(f"Trend slope: {result.trend_slope:.4f}")
print(f"Seasonality components: {result.seasonality_components}")
print(f"Metrics: {result.metrics}")
print(f"\nForecast head:")
result.forecast.head()

## Historical + Forecast with Confidence Bands

In [ ]:
fc = result.forecast

fig, ax = plt.subplots(figsize=(14, 5))

# Historical
ax.plot(df["ds"], df["y"], lw=0.8, color="steelblue", label="Historical")

# Forecast
ax.plot(fc["ds"], fc["yhat"], lw=2, color="darkorange", label="Forecast")
ax.fill_between(
    fc["ds"], fc["yhat_lower"], fc["yhat_upper"],
    alpha=0.2, color="darkorange", label="95% CI",
)

# Divider
ax.axvline(df["ds"].max(), color="gray", ls="--", lw=0.8, label="Forecast start")

ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.set_title(f"Prophet Forecast - {horizon}-Day Horizon")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Trend Decomposition

Separate the signal into trend, yearly seasonality, and weekly seasonality.

In [ ]:
# Manual decomposition for visualization
t = np.arange(len(df))
slope, intercept = np.polyfit(t, df["y"].values, 1)
trend_line = slope * t + intercept
detrended = df["y"].values - trend_line

# Extract yearly component via rolling average
yearly_component = pd.Series(detrended).rolling(window=7, center=True).mean().fillna(0).values
weekly_residual = detrended - yearly_component

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df["ds"], df["y"], lw=0.8, color="steelblue")
axes[0].set_title("Original Signal")
axes[0].set_ylabel("Value")

axes[1].plot(df["ds"], trend_line, lw=1.5, color="tomato")
axes[1].set_title(f"Linear Trend (slope = {slope:.3f} / day)")
axes[1].set_ylabel("Trend")

axes[2].plot(df["ds"], yearly_component, lw=0.8, color="green")
axes[2].set_title("Yearly Seasonality (7-day smoothed)")
axes[2].set_ylabel("Seasonal")

axes[3].plot(df["ds"], weekly_residual, lw=0.6, color="purple", alpha=0.7)
axes[3].set_title("Weekly Seasonality + Residual")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")

for ax in axes:
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
plt.show()

## Key Takeaways

- ProphetForecaster handles trend + multi-period seasonality automatically
- The 95% confidence interval widens as the forecast horizon extends
- When Prophet is not installed, the fallback model uses linear trend + daily pattern extraction
- For production use, install Prophet: `pip install -e ".[forecast]"`